# **PHẦN 4: HUẤN LUYỆN MÔ HÌNH**
## **1. Định nghĩa vấn đề**
+ **Mô tả**:
   - Huấn luyện và đánh giá hiệu suất học của mô hình
+ **Mục tiêu**:
   - Huấn luyện và xuất ra file mô hình

## **2. Chuẩn bị vấn đề**

### 2.1. Import các thư viện cần thiết

In [55]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pyarrow
import math
import itertools

from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler, RobustScaler

pd.set_option("display.max_columns", 500)
pd.set_option("display.max_rows", 500)

### 2.2. Lấy tập dữ liệu đã xử lý

In [56]:
dataFrame = pd.read_parquet("../data_processed/features.parquet")
dataFrame.head()

,Source Port,Destination Port,Protocol,Hour,Flow Duration,Total Fwd Packets,Total Length of Fwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Bwd Packet Length Max,Bwd Packet Length Min,Bwd Packet Length Mean,Flow Bytes/s,Flow Packets/s,Flow IAT Mean,Flow IAT Std,Fwd IAT Total,Fwd IAT Std,Bwd IAT Total,Bwd IAT Mean,Bwd IAT Std,Bwd IAT Min,Fwd PSH Flags,Fwd Header Length,Bwd Packets/s,Min Packet Length,FIN Flag Count,PSH Flag Count,ACK Flag Count,URG Flag Count,Down/Up Ratio,Init_Win_bytes_backward,min_seg_size_forward,Active Mean,Active Std,Idle Mean,Idle Std,Label,s_bit_0,s_bit_1,s_bit_2,s_bit_3,s_bit_4,s_bit_5,s_bit_6,s_bit_7,s_bit_8,s_bit_9,s_bit_10,s_bit_11,s_bit_12,s_bit_13,s_bit_14,s_bit_15,s_bit_16,s_bit_17,s_bit_18,s_bit_19,s_bit_20,s_bit_21,s_bit_22,s_bit_23,s_bit_24,s_bit_25,s_bit_26,s_bit_27,s_bit_28,s_bit_29,s_bit_30,s_bit_31,d_bit_0,d_bit_1,d_bit_2,d_bit_3,d_bit_4,d_bit_5,d_bit_6,d_bit_7,d_bit_8,d_bit_9,d_bit_10,d_bit_11,d_bit_12,d_bit_13,d_bit_14,d_bit_15,d_bit_16,d_bit_17,d_bit_18,d_bit_19,d_bit_20,d_bit_21,d_bit_22,d_bit_23,d_bit_24,d_bit_25,d_bit_26,d_bit_27,d_bit_28,d_bit_29,d_bit_30,d_bit_31
0,50004,443.0,6.0,11,37027,1,0.0,0.0,0.0,0.000,0.0,0.0,0.000000,0.000000,54.014638,3.702700e+04,0.000000e+00,0.0,0.000000e+00,0.0,0.000000,0.000000,0.0,0.0,32,27.007319,0.0,0.0,0.0,1.0,1.0,1,362.0,32.0,0.0,0.0,0.0,0.0,Android_Adware,0,0,0,0,1,0,1,0,0,0,1,0,1,0,1,0,0,0,0,0,0,0,0,0,1,1,0,1,0,0,1,1,1,0,1,0,1,1,0,0,1,1,0,1,1,0,0,1,0,0,0,0,0,1,1,0,1,1,0,0,1,0,1,0
1,35455,443.0,6.0,11,36653,1,0.0,0.0,0.0,0.000,0.0,0.0,0.000000,0.000000,54.565793,3.665300e+04,0.000000e+00,0.0,0.000000e+00,0.0,0.000000,0.000000,0.0,0.0,32,27.282896,0.0,0.0,0.0,1.0,1.0,1,362.0,32.0,0.0,0.0,0.0,0.0,Android_Adware,0,0,0,0,1,0,1,0,0,0,1,0,1,0,1,0,0,0,0,0,0,0,0,0,1,1,0,1,0,0,1,1,1,0,1,0,1,1,0,0,1,1,0,1,1,0,0,1,0,0,0,0,0,1,1,0,1,1,0,0,1,0,1,0
2,51775,443.0,6.0,11,534099,8,1011.0,581.0,0.0,126.375,1460.0,0.0,993.666667,24218.356522,37.446241,2.811047e+04,4.314810e+04,481340.0,6.237618e+04,487990.0,44362.727273,86342.042540,8.0,0.0,180,22.467745,0.0,0.0,1.0,0.0,0.0,1,63441.0,20.0,0.0,0.0,0.0,0.0,Android_Adware,0,0,0,0,1,0,1,0,0,0,1,0,1,0,1,0,0,0,0,0,0,0,0,0,1,1,0,1,0,0,1,1,1,0,0,0,0,0,1,1,1,1,1,1,1,1,0,1,0,0,1,1,1,1,0,1,0,1,0,0,0,1,0,0
3,51775,443.0,6.0,11,9309,3,0.0,0.0,0.0,0.000,0.0,0.0,0.000000,0.000000,322.268772,4.654500e+03,5.137131e+03,9309.0,5.137131e+03,0.0,0.000000,0.000000,0.0,0.0,60,0.000000,0.0,0.0,0.0,1.0,0.0,0,-1.0,20.0,0.0,0.0,0.0,0.0,Android_Adware,0,0,0,0,1,0,1,0,0,0,1,0,1,0,1,0,0,0,0,0,0,0,0,0,1,1,0,1,0,0,1,1,1,0,0,0,0,0,1,1,1,1,1,1,1,1,0,1,0,0,1,1,1,1,0,1,0,1,0,0,0,1,0,0
4,51776,443.0,6.0,11,19890496,8,430.0,218.0,0.0,53.750,1460.0,0.0,946.500000,307.131607,0.703854,1.530038e+06,5.377887e+06,19890496.0,7.314093e+06,410964.0,82192.800000,154845.683018,7.0,0.0,180,0.301652,0.0,0.0,1.0,0.0,0.0,0,64022.0,20.0,0.0,0.0,0.0,0.0,Android_Adware,0,0,0,0,1,0,1,0,0,0,1,0,1,0,1,0,0,0,0,0,0,0,0,0,1,1,0,1,0,0,1,1,1,0,0,0,0,0,1,1,1,1,1,1,1,1,0,1,0,0,1,1,1,1,0,1,0,1,0,0,0,1,0,0


## **3. Thực hiện vấn đề**

In [57]:
num_cols = dataFrame.select_dtypes(include="number")

filtered = num_cols.loc[:, 
    (num_cols.nunique() < 100)
]

filtered.head()

,Protocol,Hour,Fwd PSH Flags,Min Packet Length,FIN Flag Count,PSH Flag Count,ACK Flag Count,URG Flag Count,Down/Up Ratio,min_seg_size_forward,s_bit_0,s_bit_1,s_bit_2,s_bit_3,s_bit_4,s_bit_5,s_bit_6,s_bit_7,s_bit_8,s_bit_9,s_bit_10,s_bit_11,s_bit_12,s_bit_13,s_bit_14,s_bit_15,s_bit_16,s_bit_17,s_bit_18,s_bit_19,s_bit_20,s_bit_21,s_bit_22,s_bit_23,s_bit_24,s_bit_25,s_bit_26,s_bit_27,s_bit_28,s_bit_29,s_bit_30,s_bit_31,d_bit_0,d_bit_1,d_bit_2,d_bit_3,d_bit_4,d_bit_5,d_bit_6,d_bit_7,d_bit_8,d_bit_9,d_bit_10,d_bit_11,d_bit_12,d_bit_13,d_bit_14,d_bit_15,d_bit_16,d_bit_17,d_bit_18,d_bit_19,d_bit_20,d_bit_21,d_bit_22,d_bit_23,d_bit_24,d_bit_25,d_bit_26,d_bit_27,d_bit_28,d_bit_29,d_bit_30,d_bit_31
0,6.0,11,0.0,0.0,0.0,0.0,1.0,1.0,1,32.0,0,0,0,0,1,0,1,0,0,0,1,0,1,0,1,0,0,0,0,0,0,0,0,0,1,1,0,1,0,0,1,1,1,0,1,0,1,1,0,0,1,1,0,1,1,0,0,1,0,0,0,0,0,1,1,0,1,1,0,0,1,0,1,0
1,6.0,11,0.0,0.0,0.0,0.0,1.0,1.0,1,32.0,0,0,0,0,1,0,1,0,0,0,1,0,1,0,1,0,0,0,0,0,0,0,0,0,1,1,0,1,0,0,1,1,1,0,1,0,1,1,0,0,1,1,0,1,1,0,0,1,0,0,0,0,0,1,1,0,1,1,0,0,1,0,1,0
2,6.0,11,0.0,0.0,0.0,1.0,0.0,0.0,1,20.0,0,0,0,0,1,0,1,0,0,0,1,0,1,0,1,0,0,0,0,0,0,0,0,0,1,1,0,1,0,0,1,1,1,0,0,0,0,0,1,1,1,1,1,1,1,1,0,1,0,0,1,1,1,1,0,1,0,1,0,0,0,1,0,0
3,6.0,11,0.0,0.0,0.0,0.0,1.0,0.0,0,20.0,0,0,0,0,1,0,1,0,0,0,1,0,1,0,1,0,0,0,0,0,0,0,0,0,1,1,0,1,0,0,1,1,1,0,0,0,0,0,1,1,1,1,1,1,1,1,0,1,0,0,1,1,1,1,0,1,0,1,0,0,0,1,0,0
4,6.0,11,0.0,0.0,0.0,1.0,0.0,0.0,0,20.0,0,0,0,0,1,0,1,0,0,0,1,0,1,0,1,0,0,0,0,0,0,0,0,0,1,1,0,1,0,0,1,1,1,0,0,0,0,0,1,1,1,1,1,1,1,1,0,1,0,0,1,1,1,1,0,1,0,1,0,0,0,1,0,0


In [ ]:
cols = ["Fwd PSH Flags", "FIN Flag Count", "PSH Flag Count", "ACK Flag Count", "URG Flag Count", "Protocol"]

encoder = OneHotEncoder(sparse_output=False, dtype='int')
encoded_array = encoder.fit_transform(dataFrame[cols])
encoded_df = pd.DataFrame(
    encoded_array, 
    columns=encoder.get_feature_names_out(cols),
    index=dataFrame.index
)
dataFrame = pd.concat([dataFrame.drop(columns=cols), encoded_df], axis=1)
dataFrame.head()

,Source Port,Destination Port,Hour,Flow Duration,Total Fwd Packets,Total Length of Fwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Bwd Packet Length Max,Bwd Packet Length Min,Bwd Packet Length Mean,Flow Bytes/s,Flow Packets/s,Flow IAT Mean,Flow IAT Std,Fwd IAT Total,Fwd IAT Std,Bwd IAT Total,Bwd IAT Mean,Bwd IAT Std,Bwd IAT Min,Fwd Header Length,Bwd Packets/s,Min Packet Length,Down/Up Ratio,Init_Win_bytes_backward,min_seg_size_forward,Active Mean,Active Std,Idle Mean,Idle Std,Label,s_bit_0,s_bit_1,s_bit_2,s_bit_3,s_bit_4,s_bit_5,s_bit_6,s_bit_7,s_bit_8,s_bit_9,s_bit_10,s_bit_11,s_bit_12,s_bit_13,s_bit_14,s_bit_15,s_bit_16,s_bit_17,s_bit_18,s_bit_19,s_bit_20,s_bit_21,s_bit_22,s_bit_23,s_bit_24,s_bit_25,s_bit_26,s_bit_27,s_bit_28,s_bit_29,s_bit_30,s_bit_31,d_bit_0,d_bit_1,d_bit_2,d_bit_3,d_bit_4,d_bit_5,d_bit_6,d_bit_7,d_bit_8,d_bit_9,d_bit_10,d_bit_11,d_bit_12,d_bit_13,d_bit_14,d_bit_15,d_bit_16,d_bit_17,d_bit_18,d_bit_19,d_bit_20,d_bit_21,d_bit_22,d_bit_23,d_bit_24,d_bit_25,d_bit_26,d_bit_27,d_bit_28,d_bit_29,d_bit_30,d_bit_31,Fwd PSH Flags_0.0,Fwd PSH Flags_1.0,FIN Flag Count_0.0,FIN Flag Count_1.0,PSH Flag Count_0.0,PSH Flag Count_1.0,ACK Flag Count_0.0,ACK Flag Count_1.0,URG Flag Count_0.0,URG Flag Count_1.0,Protocol_0.0,Protocol_6.0,Protocol_17.0
0,50004,443.0,11,37027,1,0.0,0.0,0.0,0.000,0.0,0.0,0.000000,0.000000,54.014638,3.702700e+04,0.000000e+00,0.0,0.000000e+00,0.0,0.000000,0.000000,0.0,32,27.007319,0.0,1,362.0,32.0,0.0,0.0,0.0,0.0,Android_Adware,0,0,0,0,1,0,1,0,0,0,1,0,1,0,1,0,0,0,0,0,0,0,0,0,1,1,0,1,0,0,1,1,1,0,1,0,1,1,0,0,1,1,0,1,1,0,0,1,0,0,0,0,0,1,1,0,1,1,0,0,1,0,1,0,1,0,1,0,1,0,0,1,0,1,0,1,0
1,35455,443.0,11,36653,1,0.0,0.0,0.0,0.000,0.0,0.0,0.000000,0.000000,54.565793,3.665300e+04,0.000000e+00,0.0,0.000000e+00,0.0,0.000000,0.000000,0.0,32,27.282896,0.0,1,362.0,32.0,0.0,0.0,0.0,0.0,Android_Adware,0,0,0,0,1,0,1,0,0,0,1,0,1,0,1,0,0,0,0,0,0,0,0,0,1,1,0,1,0,0,1,1,1,0,1,0,1,1,0,0,1,1,0,1,1,0,0,1,0,0,0,0,0,1,1,0,1,1,0,0,1,0,1,0,1,0,1,0,1,0,0,1,0,1,0,1,0
2,51775,443.0,11,534099,8,1011.0,581.0,0.0,126.375,1460.0,0.0,993.666667,24218.356522,37.446241,2.811047e+04,4.314810e+04,481340.0,6.237618e+04,487990.0,44362.727273,86342.042540,8.0,180,22.467745,0.0,1,63441.0,20.0,0.0,0.0,0.0,0.0,Android_Adware,0,0,0,0,1,0,1,0,0,0,1,0,1,0,1,0,0,0,0,0,0,0,0,0,1,1,0,1,0,0,1,1,1,0,0,0,0,0,1,1,1,1,1,1,1,1,0,1,0,0,1,1,1,1,0,1,0,1,0,0,0,1,0,0,1,0,1,0,0,1,1,0,1,0,0,1,0
3,51775,443.0,11,9309,3,0.0,0.0,0.0,0.000,0.0,0.0,0.000000,0.000000,322.268772,4.654500e+03,5.137131e+03,9309.0,5.137131e+03,0.0,0.000000,0.000000,0.0,60,0.000000,0.0,0,-1.0,20.0,0.0,0.0,0.0,0.0,Android_Adware,0,0,0,0,1,0,1,0,0,0,1,0,1,0,1,0,0,0,0,0,0,0,0,0,1,1,0,1,0,0,1,1,1,0,0,0,0,0,1,1,1,1,1,1,1,1,0,1,0,0,1,1,1,1,0,1,0,1,0,0,0,1,0,0,1,0,1,0,1,0,0,1,1,0,0,1,0
4,51776,443.0,11,19890496,8,430.0,218.0,0.0,53.750,1460.0,0.0,946.500000,307.131607,0.703854,1.530038e+06,5.377887e+06,19890496.0,7.314093e+06,410964.0,82192.800000,154845.683018,7.0,180,0.301652,0.0,0,64022.0,20.0,0.0,0.0,0.0,0.0,Android_Adware,0,0,0,0,1,0,1,0,0,0,1,0,1,0,1,0,0,0,0,0,0,0,0,0,1,1,0,1,0,0,1,1,1,0,0,0,0,0,1,1,1,1,1,1,1,1,0,1,0,0,1,1,1,1,0,1,0,1,0,0,0,1,0,0,1,0,1,0,0,1,1,0,1,0,0,1,0
